In [1]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName('PIVOT')\
.config('spark.sql.repl.eagerEval.enabled', True).getOrCreate()
spark

In [11]:
df = spark.read.parquet('.//data/DATASETS/DATASETS/COMPRAS.parquet')
df.head(10)


[Row(id='012389', cartao_data_expiracao='11/25', cartao_numero='5500804500517692', cartao_bandeira='Discover', cartao_cvc='959', codigo_transacao_bancaria='GB98MPIH62210859391317', data=datetime.date(2021, 7, 24), hora='03:21:28', ipv4='62.145.31.164', ipv6='b23d:58a2:9eff:3667:47fe:d0bf:9241:433f', cep_entrega='36629-219', cd_livro='030334762', cd_cliente='3339828'),
 Row(id='012476', cartao_data_expiracao='01/31', cartao_numero='4609489235873', cartao_bandeira='VISA 16 digit', cartao_cvc='6979', codigo_transacao_bancaria='GB79GAVL23301081997574', data=datetime.date(2021, 8, 23), hora='11:15:52', ipv4='185.150.224.52', ipv6='1e2c:f641:49d1:5ae9:6d41:4e4d:2dde:5f83', cep_entrega='17012-747', cd_livro='013721981', cd_cliente='7624624'),
 Row(id='012478', cartao_data_expiracao='07/23', cartao_numero='30072722359174', cartao_bandeira='American Express', cartao_cvc='689', codigo_transacao_bancaria='GB51SZOU53848453177346', data=datetime.date(2020, 1, 27), hora='17:08:10', ipv4='40.179.153.

In [15]:
(
    df
    .withColumn('mes', F.date_format('data', 'MMMM'))
    .groupBy('mes')
    .pivot('cartao_bandeira')
    .agg(F.count('*'))
)

mes,American Express,Diners Club / Carte Blanche,Discover,JCB 15 digit,JCB 16 digit,Maestro,Mastercard,VISA 13 digit,VISA 16 digit,VISA 19 digit
July,287,295,329,342,641,281,307,324,656,352
November,310,332,301,269,623,297,298,302,616,287
February,353,395,406,374,756,374,359,350,772,388
January,385,397,454,386,835,424,415,420,878,437
March,376,367,386,401,708,378,395,406,791,385
October,309,328,306,332,615,331,337,286,685,324
May,346,291,326,337,618,324,310,314,609,331
August,303,304,339,272,618,302,289,335,632,328
April,324,300,277,311,633,319,295,299,611,315
June,291,331,297,310,596,324,337,309,620,315


TRANSFORMANDO EM TABELA UM PIVOT TABLE

In [32]:
df2 = (
    df
    .withColumn('mes', F.date_format('data', 'MMMM'))
    .groupBy('cartao_bandeira')
    .pivot('mes', ['January', 'February'])
    .agg(F.count('*'))
)


In [33]:
df2.select('cartao_bandeira', F.expr('stack(2, "Jan", January, "Fev", February) as (mes, valor)'))

cartao_bandeira,mes,valor
VISA 16 digit,Jan,878
VISA 16 digit,Fev,772
VISA 13 digit,Jan,420
VISA 13 digit,Fev,350
Discover,Jan,454
Discover,Fev,406
Diners Club / Car...,Jan,397
Diners Club / Car...,Fev,395
American Express,Jan,385
American Express,Fev,353
